In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
import shap

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"
columns = [
    "Status", "Duration", "CreditHistory", "Purpose", "CreditAmount",
    "Savings", "Employment", "InstallmentRate", "PersonalStatusSex",
    "OtherDebtors", "ResidenceDuration", "Property", "Age",
    "OtherInstallmentPlans", "Housing", "ExistingCredits",
    "Job", "Dependents", "Telephone", "ForeignWorker", "Target"
]
df = pd.read_csv(url, sep=" ", names=columns)

In [ ]:
df["Target"] = df["Target"].map({1: 0, 2: 1})

In [ ]:
X = df.drop("Target", axis=1)
y = df["Target"]
categorical_features = X.select_dtypes(include="object").columns
numerical_features = X.select_dtypes(exclude="object").columns

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42
)
pipe = Pipeline([
    ("preprocess", preprocessor),
    ("model", model)
])
pipe.fit(X_train, y_train)

In [ ]:
y_pred = pipe.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
X_train_transformed = pipe.named_steps["preprocess"].transform(X_train)
X_test_transformed = pipe.named_steps["preprocess"].transform(X_test)
if hasattr(X_train_transformed, 'toarray'): X_train_transformed = X_train_transformed.toarray()
if hasattr(X_test_transformed, 'toarray'): X_test_transformed = X_test_transformed.toarray()

feature_names = pipe.named_steps["preprocess"].get_feature_names_out()
X_test_df = pd.DataFrame(X_test_transformed, columns=feature_names)

explainer = shap.TreeExplainer(pipe.named_steps["model"])
sh_v = explainer.shap_values(X_test_df, check_additivity=False)

if isinstance(sh_v, list):
    shap_values = sh_v[1]
    base_value = explainer.expected_value[1]
elif len(sh_v.shape) == 3:
    shap_values = sh_v[:, :, 1]
    base_value = explainer.expected_value[1]
else:
    shap_values = sh_v
    base_value = explainer.expected_value

In [ ]:
shap.summary_plot(
    shap_values,
    X_test_df
)

In [ ]:
shap.summary_plot(
    shap_values,
    X_test_df,
    plot_type="bar"
)

In [ ]:
indices = [0, 5, 10]

In [ ]:
for i in indices:
    shap.plots.waterfall(
        shap.Explanation(
            values=shap_values[i],
            base_values=base_value,
            data=X_test_transformed[i],
            feature_names=feature_names
        )
    )

In [ ]:
shap.initjs()
shap.force_plot(
    base_value,
    shap_values[indices[0]],
    features=X_test_transformed[indices[0]],
    feature_names=feature_names
)